# 🐍 Day 09 · Python Basics — Sets, Sorting & Scores (Advanced Retrieval)

> **Goal of this notebook:** Advanced retrieval is mostly *counting, scoring, ranking and merging* lists. Learn sets, `Counter`, sorting and build Reciprocal Rank Fusion by hand.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. Sets — unique items
2. Counting words with `Counter`
3. Sorting with `key=`
4. A keyword score (the idea behind BM25)
5. Multi-query retrieval and removing duplicates
6. Reciprocal Rank Fusion (RRF)
7. `zip()` and `round()`

## 1. Sets — a collection of **unique** items
A set uses `{ }` like a dictionary, but stores only values. Duplicates disappear automatically.

In [ ]:
results_a = {"doc1", "doc2", "doc3"}
results_b = {"doc2", "doc3", "doc4"}

print(results_a | results_b)     # union: in A or B
print(results_a & results_b)     # intersection: in both
print(results_a - results_b)     # difference: only in A

print(set(["tesla", "nvidia", "tesla"]))   # duplicates removed

## 2. Counting words with `Counter`

In [ ]:
from collections import Counter

text = "gpu gpu ai cloud gpu ai"
counts = Counter(text.split())
print(counts)
print(counts["gpu"])               # 3
print(counts.most_common(2))       # the 2 most frequent words

## 3. Sorting with `key=`
`sorted(items, key=function)` sorts by whatever the function returns.

In [ ]:
docs = [
    {"id": "doc1", "score": 0.42},
    {"id": "doc2", "score": 0.91},
    {"id": "doc3", "score": 0.67},
]

best_first = sorted(docs, key=lambda d: d["score"], reverse=True)
for d in best_first:
    print(d["id"], d["score"])

print("Best:", max(docs, key=lambda d: d["score"])["id"])

## 4. A keyword score (the idea behind BM25)
**Keyword search** (BM25) gives a high score when the question's words appear often in a document.
**Vector search** finds documents with a similar *meaning*. **Hybrid search** uses both!

In [ ]:
documents = {
    "doc1": "nvidia makes gpu chips for ai training",
    "doc2": "tesla builds electric cars and batteries",
    "doc3": "ai models need a lot of gpu power",
}

def keyword_score(question, text):
    words = Counter(text.split())
    return sum(words[w] for w in question.split())    # add up how often each question word appears

question = "gpu for ai"
for doc_id, text in documents.items():
    print(doc_id, keyword_score(question, text))

## 5. Multi-query retrieval
`10_multi_query_retrieval.py` asks the same question in **several ways** and combines the results. A set removes the duplicates.

In [ ]:
def tiny_search(query):
    # return ids of documents that contain at least one query word
    return [d for d, t in documents.items() if set(query.split()) & set(t.split())]

queries = ["gpu for ai", "chips used to train models", "electric cars"]

all_found = set()
for q in queries:
    found = tiny_search(q)
    print(f"{q:<28} -> {found}")
    all_found.update(found)          # add to the set (duplicates ignored)

print("Unique documents:", sorted(all_found))

## 6. 🔗 Reciprocal Rank Fusion (RRF)
`11_reciprocal_rank_fusion.py` merges several ranked lists. Each document gets points `1 / (k + rank)` from every list, then we sort by total points.
Documents that appear near the top of **many** lists win.

In [ ]:
ranked_lists = [
    ["doc1", "doc3", "doc2"],    # results for query 1 (best first)
    ["doc3", "doc1"],            # results for query 2
    ["doc3", "doc2"],            # results for query 3
]
k = 60                           # a standard constant from the RRF paper

scores = {}
for ranking in ranked_lists:
    for rank, doc_id in enumerate(ranking, start=1):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)   # .get(x, 0) -> 0 the first time

final = sorted(scores.items(), key=lambda pair: pair[1], reverse=True)
for doc_id, s in final:
    print(doc_id, round(s, 4))

## 7. `zip()` — walk through two lists together

In [ ]:
names = ["vector", "keyword", "hybrid"]
accuracy = [0.71, 0.64, 0.83]

for name, acc in zip(names, accuracy):
    print(f"{name:<8} {acc:.0%}")

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Find the words that appear in BOTH `"the cat sat"` and `"the dog sat"` using sets.
2. Use `Counter` to find the most common letter in `"retrieval"`.
3. Add a 4th ranked list `["doc2", "doc1"]` to the RRF example and see if the winner changes.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
print(set("the cat sat".split()) & set("the dog sat".split()))   # {'the', 'sat'}

print(Counter("retrieval").most_common(1))

ranked_lists.append(["doc2", "doc1"])
# ...then run the RRF cell again
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Set | `{'a', 'b'}` |
| Union / both | `a \| b, a & b` |
| Count items | `Counter(words).most_common(2)` |
| Sort by key | `sorted(d, key=lambda x: x['score'])` |
| Biggest item | `max(docs, key=...)` |
| Add-or-start counter | `scores[k] = scores.get(k, 0) + 1` |
| Loop two lists | `for a, b in zip(x, y):` |

## ➡️ What's next?
You now know the Python used in **Day_09_RAG_Advanced_Retrieval**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀